# XGBoost Base Model Forecasting

This notebook implements a **Baseline XGBoost Model** for pollutant forecasting.
It serves as a comparison to the main LightGBM pipeline.

In [ ]:
import pandas as pd
import numpy as np
import warnings
import sys
import os

sys.path.append(os.getcwd())

import ispu_calculator
from ispu_calculator import calculate_ispu_for_dataframe, map_to_3_categories

from darts import TimeSeries
from darts.models import XGBModel

warnings.filterwarnings('ignore')

In [ ]:
# --- Configuration ---
POLLUTANTS = ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2']
WEATHER_FEATURES = [
    'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
    'precipitation_sum', 'wind_speed_10m_max', 'wind_speed_10m_mean',
    'relative_humidity_2m_mean', 'cloud_cover_mean', 'surface_pressure_mean'
]
STATIONS = ['DKI1', 'DKI2', 'DKI3', 'DKI4', 'DKI5']

INPUT_CHUNK = 21
OUTPUT_CHUNK = 1
FORECAST_HORIZON = 91

In [ ]:
def create_series_per_station(df, value_cols, station):
    df_st = df[df['stasiun'] == station].copy()
    df_st = df_st.sort_values('tanggal')
    df_st = df_st.groupby('tanggal')[value_cols].mean()
    df_st = df_st.asfreq('D')
    df_st = df_st.ffill().bfill()
    return TimeSeries.from_dataframe(df_st, fill_missing_dates=True, freq='D')

## 1. Load & Prepare Data

In [ ]:
try:
    df_train = pd.read_csv('../feature/final_feature.csv', parse_dates=['tanggal'])
    df_forecast = pd.read_csv('../feature/forecast_features_sep_nov_2025.csv', parse_dates=['tanggal'])
    
    df_train = df_train[df_train['stasiun'].isin(STATIONS)].copy()
    # Create series
    full_train_list = []
    full_cov_list = []
    
    for st in STATIONS:
        ts_target = create_series_per_station(df_train, POLLUTANTS, st)
        ts_cov = create_series_per_station(df_train, WEATHER_FEATURES, st)
        ts_fut = create_series_per_station(df_forecast, WEATHER_FEATURES, st)
        
        full_train_list.append(ts_target)
        full_cov_list.append(ts_cov.append(ts_fut))
        
except FileNotFoundError:
    print("Error: Data files not found.")

## 2. Train XGBoost

In [ ]:
print("Training XGBoost Model...")
model = XGBModel(
    lags=INPUT_CHUNK,
    lags_future_covariates=(INPUT_CHUNK, OUTPUT_CHUNK),
    output_chunk_length=OUTPUT_CHUNK
)

model.fit(series=full_train_list, future_covariates=full_cov_list)

## 3. Predict & Validated

In [ ]:
preds = model.predict(n=FORECAST_HORIZON, series=full_train_list, future_covariates=full_cov_list)
print("Done.")